# TNC Title + Abstract Relavance Prediction Tool

Welcome! This tool is designed to take your title + abstract data in a csv format and use a pre-trained machine learning model to predict whether each piece of text is 'Relevant' (1) or 'Irrelevant' (0). To get started, please run the following cell. It will kill your runntime after running, but don't worry as that is intended. Just move on to the next steps.


In [ ]:
# This cell installs the required Python libraries.
print("Installing required libraries... This may take a minute.")
import os
!pip install -q --upgrade --force-reinstall \
    numpy==1.23.5 \
    tensorflow==2.12.1 \
    tensorflow-hub==0.13.0 \
    tensorflow-text==2.12.1
print("Libraries installed.")

os.kill(os.getpid(), 9)

Installing required libraries... This may take a minute.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.40.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.3, but you have requests 2.32.4 which is incompatible.
altair 5.5.0 requires typing-extensions>=4.10.0; python_version < "3.14", but you have typing-extensions 4.5.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
t

## How to Use This Tool

### Step 1: Upload Your Files

You need to upload your data file (csv).

1.  **Find the File Browser:** On the left side of this Colab window, click the **folder icon**. This will open the file browser.
2.  **Upload Your Data CSV:**
    * Click the **'Upload to session storage'** button (the icon of a page with an upward arrow).
    * Select the `.csv` file from your computer that contains your title + abstract data.

Once uploaded, you should see your CSV file in the file browser.

### Step 2: Set Your Parameters

You need to tell the tool which model to use and where to find your data.

**Action:** Carefully edit the variables in the code cell below, then press run.

In [ ]:
# 1. CHOOSE YOUR MODEL
# Options are: 'SPECTER-1LAYER', 'SPECTER-3LAYER', 'BERT', 'MINBERT', 'SCIBERT', 'NAIVE BAYES'
# Make sure to type the name exactly as shown, inside the quotes.
MODEL_CHOICE = 'BERT'

# 2. PROVIDE THE PATH TO YOUR UPLOADED CSV FILE
# Replace 'your_data.csv' with the actual name of the file you uploaded.
UPLOADED_CSV_PATH = '/content/TAB_binaryLabel.csv'

# 3. SPECIFY THE NAME OF THE COLUMN CONTAINING THE TITLE + ABSTRACT
# Look at your CSV file and find the column header for the title + abstract you want to analyze.
TEXT_COLUMN_NAME = 'TAB'

# 4. PROVIDE THE NAME YOU WOULD LIKE YOUR FINAL CSV FILE TO BE NAMED
OUTPUT_FILENAME = 'predictions_output.csv'

print("Configuration set.")
print(f"Model selected: {MODEL_CHOICE}")
print(f"Data file: {UPLOADED_CSV_PATH}")

Configuration set.
Model selected: BERT
Data file: /content/TAB_binaryLabel.csv


### Step 2: Run the Following Helper Functions

3. **Download Your Model:**
    * The following lines of code help to prepare downloading the model from GitHub and setting up any additional preperations based on your previous selections.

In [ ]:
ZIPPED_FILENAME = {"SPECTER-1LAYER": "specter-1layer-model.zip",
                   "SPECTER-3LAYER": "specter-3layer-model.zip",
                   "BERT": "bert_precision.zip",
                   "MINIBERT": "minibert_model.zip",
                   "SCIBERT": "scibert_model.zip",
                   "NAIVE BAYES": "naivebayes_model.zip"
                   }

GITHUB_URL = {"SPECTER-1LAYER": "https://github.com/cia-group/tabforest/raw/main/saved_models/specter-1layer-model.zip",
              "SPECTER-3LAYER": "https://github.com/cia-group/tabforest/raw/main/saved_models/specter-3layer-model.zip",
              "BERT": "https://github.com/cia-group/tabforest/raw/main/wzheng/bert_precision.zip",
              "MINIBERT": "https://github.com/cia-group/tabforest/raw/main/saved_models/minibert_model.zip",
              "SCIBERT": "https://github.com/cia-group/tabforest/raw/main/saved_models/scibert_model.zip",
              "NAIVE BAYES": "https://github.com/cia-group/tabforest/raw/main/saved_models/naivebayes_model.zip"
              }

UNZIPPED_DIR = {"SPECTER-1LAYER": "content/specter-1layer",
                "SPECTER-3LAYER": "content/specter-3layer",
                "BERT": "/content/bert_precision",
                "MINIBERT": "/content/minibert_model",
                "SCIBERT": "/content/scibert_model",
                "NAIVE BAYES": "/content/naive_bayes_model"
                }

def retrieve_model_from_github(model_choice):
  ZIPPED_MODEL_FILENAME = ZIPPED_FILENAME.get(model_choice)
  GITHUB_MODEL_URL = GITHUB_URL.get(model_choice)
  DOWNLOAD_PATH = f'/content/{ZIPPED_MODEL_FILENAME}'
  UNZIPPED_MODEL_DIR = UNZIPPED_DIR.get(model_choice)

  if os.path.exists(UNZIPPED_MODEL_DIR):
        print(f"Model directory '{UNZIPPED_MODEL_DIR}' already exists. Skipping download and unzip.")
  else:
        print(f"Downloading '{ZIPPED_MODEL_FILENAME}' from GitHub...")
        try:
            response = requests.get(GITHUB_MODEL_URL, stream=True)
            response.raise_for_status()
            with open(DOWNLOAD_PATH, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            print("Download complete.")

            print(f"Unzipping '{ZIPPED_MODEL_FILENAME}'...")
            shutil.unpack_archive(DOWNLOAD_PATH, format='zip')
            print(f"Successfully unzipped to '{UNZIPPED_MODEL_DIR}'")

            # remove the downloaded zip file after unzipping
            os.remove(DOWNLOAD_PATH)
            print(f"Removed downloaded zip file: {DOWNLOAD_PATH}")

        except requests.exceptions.RequestException as e:
            print(f"ERROR during download: {e}")
            print(f"Could not download the file from '{GITHUB_MODEL_URL}'.")
        except FileNotFoundError:
            print(f"ERROR: Could not find the downloaded file '{DOWNLOAD_PATH}' to unzip.")
        except shutil.ReadError:
            print(f"ERROR: Could not unzip the file '{DOWNLOAD_PATH}'. It might be corrupted or not a valid zip file.")
        except Exception as e:
            print(f"An unexpected error occurred: {e}")

In [ ]:
def SPECTER_helper(example):
    """
    Helper function to embed a single example using SPECTER.
    input:: single entry from Hugging Face dataset
    return:: embed TAB column to some ex) [0.123, -0.456, 0.789, ..., 0.001]  # Shape: (768,)
    """
    from transformers import AutoModel
    model = AutoModel.from_pretrained('allenai/specter')
    model.eval() # set SPECTER tokenizer to inference mode inference mode

    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained('allenai/specter')
    input_text = example[TEXT_COLUMN_NAME]
    inputs = tokenizer(input_text, return_tensors = "pt", truncation = True, max_length = 512) # Dict {'input_ids': tensor, 'token_type_ids': tensor}

    import torch
    with torch.no_grad(): # disables gradient tracking...we are directly using SPECTER as an encoder as is, not training --> no need for backprop & store gradient
        outputs = model(**inputs) # ** unpacks the dictionary
        cls_emb = outputs.last_hidden_state[:, 0, :]  # extract CLS token...[:, 0, :] all items, first item, all items
    return {"embedding": cls_emb.squeeze().numpy()}

def encode_with_SPECTER():
  """
  Maps encode_with_SPECTER_helper to entire Hugging Face dataset.
  input:: Hugging Face dataset
  return:: Hugging Face dataset with new feature called 'embedding' w/ SPECTER embeddings, each with shape (768,)
  """
  # Change pandas dataframe to Hugging Face dataset
  from datasets import Dataset, Features, Sequence, Value
  HF_df = Dataset.from_pandas(df)

  # Add 'embedding' feature as a sequence of floats
  features = HF_df.features.copy()
  features["embedding"] = Sequence(Value("float32"))

  # Embed TAB
  tokenized_df = HF_df.map(
      SPECTER_helper,
      features = features,
      batched = False)
  print("Embedding complete.")
  return tokenized_df


In [ ]:
MODEL_FOLDER_PATHS = {"SPECTER-1LAYER": "content/specter-1layer.pt",
                      "SPECTER-3LAYER": "content/specter-3layer.pt",
                      "BERT": "/content/bert_precision_hyperparam",
                      "MINIBERT": "/content/minibert_model",
                      "SCIBERT": "/content/scibert_model",
                      "NAIVE BAYES": "/content/naivebayes.pickle"
                }

MODEL_PACKAGE = {"SPECTER-1LAYER": "PyTorch",
                 "SPECTER-3LAYER": "PyTorch",
                 "BERT": "TensorFlow",
                 "MINIBERT": "PyTorch",
                 "SCIBERT": "PyTorch",
                 "NAIVE BAYES": "Scikit-Learn"
                 }

PREDICTION_THRESHOLD = {"SPECTER-1LAYER": 0.5,
                        "SPECTER-3LAYER": 0.5,
                        "BERT": 0.8735,
                        "MINIBERT": 0.5,
                        "SCIBERT": 0.5,
                        "NAIVE BAYES": 0.5 # note naive bayes model only ouputs 0/1, 0.5 threshold is more of a placeholder
                        }

def make_predictions(model_choice):
  model_path = MODEL_FOLDER_PATHS.get(model_choice)
  print(f"Loading model from: {model_path}")

  # WAYS TO LOAD ALL MODELS!

  if MODEL_PACKAGE.get(model_choice) == "TensorFlow":
    # Import necessary packages
    print("Importing corresponding packages...")
    import tensorflow as tf
    import tensorflow_hub as hub
    import tensorflow_text as text
    print("Import complete.")

    # Load model
    model = tf.saved_model.load(model_path)
    model.eval()
    print("Model loaded.")

    # Make predictions
    print("Making predictions...")
    infer = model.signatures["serving_default"]
    try:
      raw_predictions = infer(text=text_to_predict)['classifier'].numpy() # Changed output key to 'classifier'
    except KeyError as e:
      print(f"KeyError: {e}")
      print("Available output keys:")
      print(infer(text=text_to_predict).keys())
      sys.exit() # Exit after printing keys to avoid further errors

  elif MODEL_PACKAGE.get(model_choice) == "PyTorch":
    # Import necessary packages
    print("Importing corresponding packages...")
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from transformers import AutoConfig, AutoModel
    print("Import complete.")

    # Load model
    if "SPECTER" in model_choice:  # Embed with SPECTER if necessary
      print("Starting TAB embedding with SPECTER...")
      encoded = encode_with_SPECTER()
      input = torch.tensor(encoded["embedding"], dtype = torch.float32)
      print("SPECTER embedding complete.")
      model = torch.load(model_path, weights_only = False)

    else:
      config = AutoConfig.from_pretrained(model_path)
      model = AutoModel.from_pretrained(model_path, config=config, trust_remote_code=True)
      input = df[TEXT_COLUMN_NAME]

    model.eval()
    print("Model loaded.")

    # Make predictions
    print("Making predictions...")
    raw_predictions = model(input)

  else: # Scikit-learn
    # Import necessary packages
    print("Importing corresponding packages...")
    import pickle
    from sklearn.feature_extraction.text import TfidfVectorizer
    print("Import complete.")

    # Load the model
    with open(model_path, "rb") as f:
      model = pickle.load(f)

    # Make predictions
    tfidf_vectorizer = TfidfVectorizer()
    input = tfidf_vectorizer.fit_transform(df[TEXT_COLUMN_NAME])
    raw_predictions = model.predict(input)


  # Add results to the dataframe
  df['prediction_score'] = raw_predictions.detach().flatten().numpy() # raw probability from the model
  df['prediction'] = (df['prediction_score'] >= PREDICTION_THRESHOLD.get(model_choice)).astype(int)

  print("\n Prediction Complete. ")
  print("\nHere is a preview of your results:")
  display(df.head())

  print("\nSummary of Predictions:")
  print(df['prediction'].astype(str).value_counts())

In [ ]:
def upload_data(UPLOADED_CSV_PATH):
  try:
    df = pd.read_csv(UPLOADED_CSV_PATH)
    # Ensure the tab column exists
    if TEXT_COLUMN_NAME not in df.columns:
      print(f"ERROR: Column '{TEXT_COLUMN_NAME}' not found in your CSV file.")
      print(f"Available columns are: {list(df.columns)}")
  except FileNotFoundError:
    print(f"ERROR: Could not find the CSV file at '{UPLOADED_CSV_PATH}'.")
    print("Please go back to Step 1 and make sure you have uploaded the file.")
  return df

### Step 3: Run the Prediction!

Now you are ready to run the model. The code below will handle everything automatically based on your settings from Step 1, from downloading the model to uploading your dataset and processing your data.

**Action:** Run the following cell. It performs the actual prediction.

In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import shutil
import requests

retrieve_model_from_github(MODEL_CHOICE)
df = pd.read_csv(UPLOADED_CSV_PATH).head(3) # put 3 for speed, otherwise df = upload_data(UPLOADED_CSV_PATH)
make_predictions(MODEL_CHOICE)

Loading data and model...
Model directory '/content/minibert_model' already exists. Skipping download and unzip.
Loading model from: /content/minibert_model
Importing corresponding packages...
Import complete.
Model loaded.
Making predictions...


KeyError: 'key of type tuple not found and not a MultiIndex'

### Step 4: Save and Download Your Results

The final step is to save your data, now with the new prediction columns, to a new CSV file that you can download to your computer.

**Action:** Run the code cell below. Your browser will automatically start downloading the file `predictions_output.csv`.

In [ ]:
from google.colab import files

OUTPUT_FILENAME = 'predictions_output.csv'

print(f"Saving results to {OUTPUT_FILENAME}...")
df.to_csv(OUTPUT_FILENAME, index=False)

print(f"Results saved. Starting download...")
files.download(output_filename)

Saving results to predictions_output.csv...
Results saved. Starting download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download complete.
